# 04 – Details

Esplorazione e data cleaning del dataset `details.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `mal_id` | ID univoco dell'anime su MyAnimeList |
| `title` | Titolo dell'anime (romaji/inglese) |
| `title_japanese` | Titolo in giapponese |
| `url` | URL della pagina MAL dell'anime |
| `image_url` | URL dell'immagine di copertina |
| `type` | Tipo (TV, Movie, OVA, ONA, Special, Music, CM, PV, TV Special) |
| `status` | Stato di trasmissione |
| `score` | Punteggio medio (1–10, float) |
| `scored_by` | Numero di utenti che hanno votato |
| `start_date` | Data di inizio trasmissione (ISO 8601) |
| `end_date` | Data di fine trasmissione (ISO 8601) |
| `synopsis` | Sinossi testuale |
| `rank` | Classifica globale MAL |
| `popularity` | Posizione nella classifica di popolarità |
| `members` | Numero di utenti che hanno l'anime nella loro lista |
| `favorites` | Numero di utenti che hanno messo l'anime tra i preferiti |
| `genres` | Lista di generi (stringa con lista Python) |
| `studios` | Lista di studi di produzione |
| `themes` | Lista di temi narrativi |
| `demographics` | Lista di target demografici |
| `source` | Materiale di origine (Manga, Novel, Original, …) |
| `rating` | Classificazione per età |
| `episodes` | Numero di episodi |
| `season` | Stagione di uscita (spring/summer/fall/winter) |
| `year` | Anno di uscita |
| `producers` | Lista di produttori |
| `explicit_genres` | Lista di generi espliciti |
| `licensors` | Lista di licenziatari |
| `streaming` | Lista di piattaforme di streaming |

## 1. Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

In [2]:
df_details = pd.read_csv('../datasets/details.csv')
print(f'Shape: {df_details.shape}')
df_details.info()
df_details.head()

Shape: (28955, 29)
<class 'pandas.DataFrame'>
RangeIndex: 28955 entries, 0 to 28954
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   mal_id           28955 non-null  int64  
 1   title            28955 non-null  str    
 2   title_japanese   28836 non-null  str    
 3   url              28955 non-null  str    
 4   image_url        28955 non-null  str    
 5   type             28888 non-null  str    
 6   status           28955 non-null  str    
 7   score            18882 non-null  float64
 8   scored_by        18882 non-null  float64
 9   start_date       28104 non-null  str    
 10  end_date         11267 non-null  str    
 11  synopsis         23828 non-null  str    
 12  rank             21997 non-null  float64
 13  popularity       28955 non-null  int64  
 14  members          28955 non-null  int64  
 15  favorites        28955 non-null  int64  
 16  genres           28955 non-null  str    
 17  stud

,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming
0,59356,-Socket-,-socket-,https://myanimelist.net/anime/59356/-Socket-,https://cdn.myanimelist.net/images/anime/1043/...,Movie,Finished Airing,NaN,NaN,2010-01-01T00:00:00+00:00,...,[],Original,G - All Ages,1.0,NaN,NaN,['Nagoya Zokei University'],[],[],[]
1,56036,......,......,https://myanimelist.net/anime/56036/-,https://cdn.myanimelist.net/images/anime/1057/...,Music,Finished Airing,6.53,503.0,2023-06-11T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,1.0,NaN,NaN,[],[],[],[]
2,2928,.hack//G.U. Returner,.HACK//G.U. RETURNER,https://myanimelist.net/anime/2928/hack__GU_Re...,https://cdn.myanimelist.net/images/anime/1798/...,OVA,Finished Airing,6.65,9745.0,2007-01-18T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,"['Bandai Visual', 'CyberConnect2']",[],[],[]
3,3269,.hack//G.U. Trilogy,.hack//G.U. Trilogy,https://myanimelist.net/anime/3269/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/1566/...,Movie,Finished Airing,7.06,15373.0,2007-12-22T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,['Bandai Visual'],[],"['Funimation', 'Bandai Entertainment']",[]
4,4469,.hack//G.U. Trilogy: Parody Mode,.hack//G.U. Trilogy,https://myanimelist.net/anime/4469/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/10/86...,Special,Finished Airing,6.35,4317.0,2008-03-25T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,['Bandai Visual'],[],[],[]


In [3]:
n_originale = len(df_details)

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [4]:
mask_dup = df_details.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                  # righe totali che hanno almeno un duplicato
n_gruppi = df_details[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi             # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_details.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_details):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 28,955
Righe dopo la rimozione     : 28,955


## 2. Analisi colonna per colonna

### 2.1 `mal_id`

In [5]:
analyze(df_details['mal_id'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'mal_id'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    mal_id
  dtype                          int64
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          0  (0.00%)
  Positives                      28,955   (100.00%)
  Negatives                      0   (0.00%)
  Unique values                  28,955  (100.00%)

 📐 Descriptive Statistics
──────────────────────────────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, nessun duplicato, dtype già `int64`. È la chiave primaria del dataset.

In [6]:
# Nessuna operazione richiesta
print(f'Null in mal_id      : {df_details["mal_id"].isna().sum()}')
print(f'Duplicati in mal_id : {df_details["mal_id"].duplicated().sum()}')

Null in mal_id      : 0
Duplicati in mal_id : 0


### 2.2 `title`

In [7]:
analyze(df_details['title'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════════  🔬  SERIES ANALYSIS  —  'title'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    title
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  28,953  (99.99%)
  Duplicate values               2

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     1  chars
  Max length  

        1.00 – 10.25      │██████████████████ 3742
       10.25 – 19.50      │█████████████████████████████████████████████ 9458
       19.50 – 28.75      │████████████████████████████████ 6669
       28.75 – 38.00      │████████████████████ 4282
       38.00 – 47.25      │████████████ 2418
       47.25 – 56.50      │██████ 1192
       56.50 – 65.75      │███ 639
       65.75 – 75.00      │█ 269
       75.00 – 84.25      │█ 154
       84.25 – 93.50      │ 61
       93.50 – 102.75     │ 30
      102.75 – 112.00     │ 23
      112.00 – 121.25     │ 10
      121.25 – 130.50     │ 4
      130.50 – 139.75     │ 1
      139.75 – 149.00     │ 2
      149.00 – 158.25     │ 0
      158.25 – 167.50     │ 0
      167.50 – 176.75     │ 0
      176.75 – 186.00     │ 1

 🔢 Most & Least Frequent Values
────────────────────────────────────────────────────────────────────────────────
  Top 10 most frequent:
    Moonlight                   [█████████████████████████] 100.0%  2
    Shen Lan Qi Yu Wushuan

**Nessuna pulizia necessaria.**  
Nessun null. I titoli duplicati sono attesi: adattamenti e remake diversi possono condividere lo stesso titolo.

In [8]:
# Nessuna operazione richiesta
print(f'Null in title : {df_details["title"].isna().sum()}')

Null in title : 0


### 2.3 `title_japanese`

In [9]:
analyze(df_details['title_japanese'])


════════════════════════════════════════════════════════════════════════════════
══════════════════  🔬  SERIES ANALYSIS  —  'title_japanese'  ═══════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    title_japanese
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,836  [██████████████████████████████]  99.6%
  Null / NaN                     119  (0.41%)
  Empty strings                  0
  Unique values                  27,772  (95.91%)
  Duplicate values               1,064

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     1  chars

**Nessuna pulizia necessaria.**  
I ~119 null sono strutturali: anime di produzione non giapponese o senza titolo kanji registrato su MAL.

In [10]:
# Nessuna operazione richiesta: i null sono strutturali
print(f'Null in title_japanese : {df_details["title_japanese"].isna().sum()} ({df_details["title_japanese"].isna().mean()*100:.1f}%)')

Null in title_japanese : 119 (0.4%)


### 2.4 `url`

In [11]:
analyze(df_details['url'])


════════════════════════════════════════════════════════════════════════════════
════════════════════════  🔬  SERIES ANALYSIS  —  'url'  ════════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    url
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  28,955  (100.00%)
  Duplicate values               0

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     35  chars
  Max length  

**Nessuna pulizia necessaria.**  
Nessun null, tutti gli URL sono unici (coerente con `mal_id` come chiave primaria).

In [12]:
# Nessuna operazione richiesta
print(f'Null in url      : {df_details["url"].isna().sum()}')
print(f'Duplicati in url : {df_details["url"].duplicated().sum()}')

Null in url      : 0
Duplicati in url : 0


### 2.5 `image_url`

In [13]:
DEFAULT_IMG = 'https://cdn.myanimelist.net/img/sp/icon/apple-touch-icon-256.png'
placeholder = (df_details['image_url'] == DEFAULT_IMG).sum()
print(f'Immagini placeholder : {placeholder:,} ({placeholder/len(df_details)*100:.1f}%)')
print(f'Immagini reali       : {len(df_details)-placeholder:,} ({(len(df_details)-placeholder)/len(df_details)*100:.1f}%)')
print()
analyze(df_details['image_url'])

Immagini placeholder : 198 (0.7%)
Immagini reali       : 28,757 (99.3%)


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'image_url'  ═════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    image_url
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  28,758  (99.32%)
  Duplicate values               197

 📐 String Length Analysis
──────────────────────────────────────────────────────────

**Pulizia necessaria:**  
~198 URL corrispondono all'immagine placeholder di MAL → aggiungere colonna booleana `has_image` per distinguere gli anime con copertina reale.

In [14]:
DEFAULT_IMG = 'https://cdn.myanimelist.net/img/sp/icon/apple-touch-icon-256.png'
df_details['has_image'] = df_details['image_url'] != DEFAULT_IMG
print(f'has_image → True (reale): {df_details["has_image"].sum():,}  |  False (placeholder): {(~df_details["has_image"]).sum():,}')

has_image → True (reale): 28,757  |  False (placeholder): 198


### 2.6 `type`

In [15]:
analyze(df_details['type'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════════  🔬  SERIES ANALYSIS  —  'type'  ════════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    type
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,888  [██████████████████████████████]  99.8%
  Null / NaN                     67  (0.23%)
  Empty strings                  0
  Unique values                  9  (0.03%)
  Duplicate values               28,879

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     2  chars
  Max length   

**Nessuna pulizia necessaria.**  
I ~67 null sono strutturali: anime con stato `Not yet aired` per cui il tipo non è ancora stato registrato su MAL. I valori presenti sono coerenti con la tassonomia MAL (TV, Movie, OVA, ONA, Special, Music, CM, PV, TV Special).

In [16]:
# Nessuna operazione richiesta: i null sono strutturali (Not yet aired)
print(f'Null in type  : {df_details["type"].isna().sum()}')
print(f'Valori unici  : {sorted(df_details["type"].dropna().unique().tolist())}')

Null in type  : 67
Valori unici  : ['CM', 'Movie', 'Music', 'ONA', 'OVA', 'PV', 'Special', 'TV', 'TV Special']


### 2.7 `status`

In [17]:
analyze(df_details['status'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'status'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    status
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  3  (0.01%)
  Duplicate values               28,952

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     13  chars
  Max length 

**Nessuna pulizia necessaria.**  
Nessun null. Tre valori coerenti: `Finished Airing`, `Currently Airing`, `Not yet aired`.

In [18]:
# Nessuna operazione richiesta
print(f'Null in status : {df_details["status"].isna().sum()}')
print(df_details['status'].value_counts())

Null in status : 0
status
Finished Airing     28097
Not yet aired         544
Currently Airing      314
Name: count, dtype: int64


### 2.8 `score` e `scored_by`

Le due colonne sono co-nulle: quando un anime non ha voti sufficienti su MAL, entrambe sono `NaN`.

In [19]:
analyze(df_details['score'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════════  🔬  SERIES ANALYSIS  —  'score'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    score
  dtype                          float64
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                18,882  [████████████████████░░░░░░░░░░]  65.2%
  Null / NaN                     10,073  (34.79%)
  Zeros                          0  (0.00%)
  Positives                      18,882   (65.21%)
  Negatives                      0   (0.00%)
  Unique values                  561  (1.94%)

 📐 Descriptive Statistics
─────────────────────────────────────────────────────────────────────

In [20]:
analyze(df_details['scored_by'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'scored_by'  ═════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    scored_by
  dtype                          float64
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                18,882  [████████████████████░░░░░░░░░░]  65.2%
  Null / NaN                     10,073  (34.79%)
  Zeros                          0  (0.00%)
  Positives                      18,882   (65.21%)
  Negatives                      0   (0.00%)
  Unique values                  9,245  (31.93%)

 📐 Descriptive Statistics
──────────────────────────────────────────────────────────────

**Pulizie necessarie:**
- I null (~10.073) sono strutturali: anime senza voti sufficienti per avere uno score su MAL → nessuna rimozione
- `scored_by`: dtype `float64` → convertire a `Int64` (nullable integer), dato che i valori sono interi

In [21]:
# Verifica co-nullità
print(f'Null in score     : {df_details["score"].isna().sum():,}')
print(f'Null in scored_by : {df_details["scored_by"].isna().sum():,}')
print(f'Co-nulli (entrambi NaN) : {(df_details["score"].isna() & df_details["scored_by"].isna()).sum():,}')
print()

df_details['scored_by'] = df_details['scored_by'].astype('Int64')
print(f'scored_by dtype : {df_details["scored_by"].dtype}')

Null in score     : 10,073
Null in scored_by : 10,073
Co-nulli (entrambi NaN) : 10,073

scored_by dtype : Int64


### 2.9 `start_date`

In [22]:
analyze(df_details['start_date'])


════════════════════════════════════════════════════════════════════════════════
════════════════════  🔬  SERIES ANALYSIS  —  'start_date'  ═════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    start_date
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,104  [█████████████████████████████░]  97.1%
  Null / NaN                     851  (2.94%)
  Empty strings                  0
  Unique values                  9,367  (32.35%)
  Duplicate values               18,737

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     25  chars
  

**Pulizia necessaria:**
- I valori sono stringhe ISO 8601 con timezone UTC → convertire a `datetime64[us, UTC]`
- Gli ~851 null sono strutturali: anime `Not yet aired` senza data confermata

In [23]:
df_details['start_date'] = pd.to_datetime(df_details['start_date'], errors='coerce', utc=True)
print(f'start_date dtype : {df_details["start_date"].dtype}')
print(f'Null residui     : {df_details["start_date"].isna().sum()}')
print(f'Range            : {df_details["start_date"].min()} → {df_details["start_date"].max()}')

start_date dtype : datetime64[us, UTC]
Null residui     : 851
Range            : 1917-01-01 00:00:00+00:00 → 2027-01-01 00:00:00+00:00


### 2.10 `end_date`

In [24]:
analyze(df_details['end_date'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'end_date'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    end_date
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                11,267  [████████████░░░░░░░░░░░░░░░░░░]  38.9%
  Null / NaN                     17,688  (61.09%)
  Empty strings                  0
  Unique values                  5,607  (19.36%)
  Duplicate values               5,660

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     25  chars
 

**Pulizia necessaria:**
- Stessa conversione a `datetime64[us, UTC]` di `start_date`
- I ~17.688 null sono strutturali: anime `Currently Airing` o `Not yet aired` non hanno ancora una data di fine

In [25]:
df_details['end_date'] = pd.to_datetime(df_details['end_date'], errors='coerce', utc=True)
print(f'end_date dtype : {df_details["end_date"].dtype}')
print(f'Null residui   : {df_details["end_date"].isna().sum()}')
print(f'Range          : {df_details["end_date"].min()} → {df_details["end_date"].max()}')

end_date dtype : datetime64[us, UTC]
Null residui   : 17688
Range          : 1962-02-25 00:00:00+00:00 → 2026-04-10 00:00:00+00:00


### 2.11 `synopsis`

In [26]:
analyze(df_details['synopsis'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'synopsis'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    synopsis
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                23,828  [█████████████████████████░░░░░]  82.3%
  Null / NaN                     5,127  (17.71%)
  Empty strings                  0
  Unique values                  23,540  (81.30%)
  Duplicate values               288

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     10  chars
  M

**Pulizia necessaria:**
- I ~5.127 null sono strutturali: anime senza sinossi su MAL (spesso cortometraggi, CM, PV) → aggiungere colonna booleana `has_synopsis`

In [27]:
df_details['has_synopsis'] = df_details['synopsis'].notna()
print(f'has_synopsis → True (con sinossi): {df_details["has_synopsis"].sum():,}  |  False: {(~df_details["has_synopsis"]).sum():,}')

has_synopsis → True (con sinossi): 23,828  |  False: 5,127


### 2.12 `rank`

In [28]:
analyze(df_details['rank'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════════  🔬  SERIES ANALYSIS  —  'rank'  ════════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    rank
  dtype                          float64
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                21,997  [███████████████████████░░░░░░░]  76.0%
  Null / NaN                     6,958  (24.03%)
  Zeros                          0  (0.00%)
  Positives                      21,997   (75.97%)
  Negatives                      0   (0.00%)
  Unique values                  17,765  (61.35%)

 📐 Descriptive Statistics
───────────────────────────────────────────────────────────────────

**Pulizia necessaria:**
- I ~6.958 null sono strutturali: anime non classificati da MAL (score assente o insufficiente)
- `dtype` float64 → convertire a `Int64` (nullable integer), dato che i valori sono interi

In [29]:
df_details['rank'] = df_details['rank'].astype('Int64')
print(f'rank dtype   : {df_details["rank"].dtype}')
print(f'Null residui : {df_details["rank"].isna().sum():,}')

rank dtype   : Int64
Null residui : 6,958


### 2.13 `popularity`, `members`, `favorites`

In [30]:
for col in ['popularity', 'members', 'favorites']:
    analyze(df_details[col])


════════════════════════════════════════════════════════════════════════════════
════════════════════  🔬  SERIES ANALYSIS  —  'popularity'  ═════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    popularity
  dtype                          int64
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          0  (0.00%)
  Positives                      28,955   (100.00%)
  Negatives                      0   (0.00%)
  Unique values                  21,954  (75.82%)

 📐 Descriptive Statistics
───────────────────────────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. Le distribuzioni sono fortemente asimmetriche (code lunghe verso destra), da tenere in considerazione nelle analisi.

In [31]:
# Nessuna operazione richiesta
for col in ['popularity', 'members', 'favorites']:
    print(f'Null in {col:<12}: {df_details[col].isna().sum()}')

Null in popularity  : 0
Null in members     : 0
Null in favorites   : 0


### 2.14 `genres`, `studios`, `themes`, `demographics`, `producers`, `licensors`, `streaming`

Colonne contenenti liste serializzate come stringhe Python (es. `"['Comedy', 'Drama']"`).

In [32]:
list_cols = ['genres', 'studios', 'themes', 'demographics', 'producers', 'licensors', 'streaming']
for col in list_cols:
    analyze(df_details[col])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'genres'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    genres
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  931  (3.22%)
  Duplicate values               28,024

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     2  chars
  Max length

**Nessuna pulizia necessaria.**  
Nessun null in nessuna di queste colonne. Le liste vuote `[]` sono legittime (anime senza studio, senza streaming, ecc.). Il formato stringa viene mantenuto così com'è: è sufficiente per le analisi testuali e può essere deserializzato con `ast.literal_eval` quando serve.

In [33]:
# Nessuna operazione richiesta
list_cols = ['genres', 'studios', 'themes', 'demographics', 'producers', 'licensors', 'streaming']
for col in list_cols:
    print(f'Null in {col:<14}: {df_details[col].isna().sum()}')

Null in genres        : 0
Null in studios       : 0
Null in themes        : 0
Null in demographics  : 0
Null in producers     : 0
Null in licensors     : 0
Null in streaming     : 0


### 2.15 `source`

In [34]:
analyze(df_details['source'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'source'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    source
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  17  (0.06%)
  Duplicate values               28,938

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     4  chars
  Max length 

**Nessuna pulizia necessaria.**  
Nessun null. I valori sono coerenti con la tassonomia MAL.

In [35]:
# Nessuna operazione richiesta
print(f'Null in source : {df_details["source"].isna().sum()}')
print(df_details['source'].value_counts())

Null in source : 0
source
Original        12359
Manga            5508
Unknown          2794
Game             1473
Other            1326
Light novel      1199
Visual novel     1160
Novel             820
Web manga         654
4-koma manga      339
Picture book      284
Music             263
Mixed media       237
Book              224
Web novel         222
Card game          79
Radio              14
Name: count, dtype: int64


### 2.16 `rating`

In [36]:
analyze(df_details['rating'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'rating'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    rating
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,355  [█████████████████████████████░]  97.9%
  Null / NaN                     600  (2.07%)
  Empty strings                  0
  Unique values                  6  (0.02%)
  Duplicate values               28,349

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     11  chars
  Max lengt

**Nessuna pulizia necessaria.**  
I ~600 null sono strutturali: anime senza classificazione per età registrata su MAL. I valori presenti (G, PG, PG-13, R-17+, R+, Rx) sono coerenti con la classificazione MAL.

In [37]:
# Nessuna operazione richiesta: i null sono strutturali
print(f'Null in rating : {df_details["rating"].isna().sum()}')
print(df_details['rating'].value_counts())

Null in rating : 600
rating
PG-13 - Teens 13 or older         10463
G - All Ages                       9066
PG - Children                      4421
R - 17+ (violence & profanity)     1593
Rx - Hentai                        1583
R+ - Mild Nudity                   1229
Name: count, dtype: int64


### 2.17 `episodes`

In [38]:
analyze(df_details['episodes'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'episodes'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    episodes
  dtype                          float64
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,275  [█████████████████████████████░]  97.7%
  Null / NaN                     680  (2.35%)
  Zeros                          0  (0.00%)
  Positives                      28,275   (97.65%)
  Negatives                      0   (0.00%)
  Unique values                  261  (0.90%)

 📐 Descriptive Statistics
──────────────────────────────────────────────────────────────────────

**Pulizia necessaria:**
- I ~680 null sono strutturali: anime in corso o con numero di episodi non ancora definito
- `dtype` float64 → convertire a `Int64` (nullable integer), dato che i valori sono interi

In [39]:
df_details['episodes'] = df_details['episodes'].astype('Int64')
print(f'episodes dtype   : {df_details["episodes"].dtype}')
print(f'Null residui     : {df_details["episodes"].isna().sum()}')
print(f'Range            : {df_details["episodes"].min()} – {df_details["episodes"].max()}')

episodes dtype   : Int64
Null residui     : 680
Range            : 1 – 3000


### 2.18 `season`

In [40]:
analyze(df_details['season'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'season'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    season
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                6,266  [██████░░░░░░░░░░░░░░░░░░░░░░░░]  21.6%
  Null / NaN                     22,689  (78.36%)
  Empty strings                  0
  Unique values                  4  (0.01%)
  Duplicate values               6,262

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     4  chars
  Max leng

**Nessuna pulizia necessaria.**  
I ~22.689 null (~78%) sono strutturali: Movies, OVA, ONA, Music, CM e PV non appartengono a una stagione MAL. I quattro valori (`spring`, `summer`, `fall`, `winter`) sono coerenti.

In [41]:
# Nessuna operazione richiesta: i null sono strutturali
print(f'Null in season : {df_details["season"].isna().sum():,} ({df_details["season"].isna().mean()*100:.1f}%)')
print(df_details['season'].value_counts())

Null in season : 22,689 (78.4%)
season
spring    1972
fall      1825
winter    1302
summer    1167
Name: count, dtype: int64


### 2.19 `year`

In [42]:
analyze(df_details['year'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════════  🔬  SERIES ANALYSIS  —  'year'  ════════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    year
  dtype                          float64
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                6,266  [██████░░░░░░░░░░░░░░░░░░░░░░░░]  21.6%
  Null / NaN                     22,689  (78.36%)
  Zeros                          0  (0.00%)
  Positives                      6,266   (21.64%)
  Negatives                      0   (0.00%)
  Unique values                  66  (0.23%)

 📐 Descriptive Statistics
─────────────────────────────────────────────────────────────────────────

**Pulizia necessaria:**
- I null sono coerenti con quelli di `season` (stessa logica strutturale)
- `dtype` float64 → convertire a `Int64` (nullable integer)

In [43]:
df_details['year'] = df_details['year'].astype('Int64')
print(f'year dtype   : {df_details["year"].dtype}')
print(f'Null residui : {df_details["year"].isna().sum():,}')
print(f'Range        : {df_details["year"].min()} – {df_details["year"].max()}')

year dtype   : Int64
Null residui : 22,689
Range        : 1961 – 2026


### 2.20 `explicit_genres`

In [44]:
analyze(df_details['explicit_genres'])


════════════════════════════════════════════════════════════════════════════════
══════════════════  🔬  SERIES ANALYSIS  —  'explicit_genres'  ══════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    explicit_genres
  dtype                          str
  Shape                          28,955
  Index range                    0 … 28954
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     28,955
  Non-null values                28,955  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  1  (0.00%)
  Duplicate values               28,954

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     2  chars
  Max

**Pulizia necessaria:**
- La colonna contiene esclusivamente il valore `[]` (lista vuota) per tutte le righe → nessuna informazione utile
- Rimozione della colonna

In [45]:
print(f'Valori unici in explicit_genres: {df_details["explicit_genres"].unique()}')
df_details.drop(columns=['explicit_genres'], inplace=True)
print('Colonna explicit_genres rimossa.')

Valori unici in explicit_genres: <StringArray>
['[]']
Length: 1, dtype: str
Colonna explicit_genres rimossa.


### 2.21 Chiave primaria `mal_id`

Verifichiamo che `mal_id` sia ancora univoco dopo tutte le operazioni di pulizia.

In [46]:
n_pk_dup = df_details.duplicated(subset=['mal_id'], keep=False).sum()
print(f'Duplicati su mal_id: {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_details.drop_duplicates(subset=['mal_id'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_details):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su mal_id: 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Data Cleaning

In [47]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_details):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_details):>10,}')
print()
print('Dtypes finali:')
print(df_details.dtypes)
print()
print('Valori mancanti residui:')
print(df_details.isnull().sum())

=== Riepilogo Dataset Pulito ===
Righe originali      :     28,955
Righe dopo cleaning  :     28,955
Righe rimosse totali :          0

Dtypes finali:
mal_id                          int64
title                             str
title_japanese                    str
url                               str
image_url                         str
type                              str
status                            str
score                         float64
scored_by                       Int64
start_date        datetime64[us, UTC]
end_date          datetime64[us, UTC]
synopsis                          str
rank                            Int64
popularity                      int64
members                         int64
favorites                       int64
genres                            str
studios                           str
themes                            str
demographics                      str
source                            str
rating                            str
episodes     

In [48]:
df_details.head()

,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,source,rating,episodes,season,year,producers,licensors,streaming,has_image,has_synopsis
0,59356,-Socket-,-socket-,https://myanimelist.net/anime/59356/-Socket-,https://cdn.myanimelist.net/images/anime/1043/...,Movie,Finished Airing,NaN,<NA>,2010-01-01 00:00:00+00:00,...,Original,G - All Ages,1,NaN,<NA>,['Nagoya Zokei University'],[],[],True,True
1,56036,......,......,https://myanimelist.net/anime/56036/-,https://cdn.myanimelist.net/images/anime/1057/...,Music,Finished Airing,6.53,503,2023-06-11 00:00:00+00:00,...,Original,PG-13 - Teens 13 or older,1,NaN,<NA>,[],[],[],True,True
2,2928,.hack//G.U. Returner,.HACK//G.U. RETURNER,https://myanimelist.net/anime/2928/hack__GU_Re...,https://cdn.myanimelist.net/images/anime/1798/...,OVA,Finished Airing,6.65,9745,2007-01-18 00:00:00+00:00,...,Game,PG-13 - Teens 13 or older,1,NaN,<NA>,"['Bandai Visual', 'CyberConnect2']",[],[],True,True
3,3269,.hack//G.U. Trilogy,.hack//G.U. Trilogy,https://myanimelist.net/anime/3269/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/1566/...,Movie,Finished Airing,7.06,15373,2007-12-22 00:00:00+00:00,...,Game,PG-13 - Teens 13 or older,1,NaN,<NA>,['Bandai Visual'],"['Funimation', 'Bandai Entertainment']",[],True,True
4,4469,.hack//G.U. Trilogy: Parody Mode,.hack//G.U. Trilogy,https://myanimelist.net/anime/4469/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/10/86...,Special,Finished Airing,6.35,4317,2008-03-25 00:00:00+00:00,...,Game,PG-13 - Teens 13 or older,1,NaN,<NA>,['Bandai Visual'],[],[],True,True


## 4. Salvataggio dataset pulito

In [49]:
df_details.to_csv('../datasets_cleaned/details_clean.csv', index=False)
print('Salvato: datasets_cleaned/details_clean.csv')

Salvato: datasets_cleaned/details_clean.csv
